<a href="https://colab.research.google.com/github/rickiepark/llm-from-scratch/blob/main/ch06/01_main-chapter-code/ch06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
세바스찬 라시카(Sebastian Raschka)가 쓴 <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a>의 번역서 <br><<b><a href="<a href="http://tensorflow.blog/llm-from-scratch">밑바닥부터 만들면서 배우는 LLM</a></b>>의 예제 코드입니다.<br>
<br>코드 저장소: <a href="https://github.com/rickiepark/llm-from-scratch">https://github.com/rickiepark/llm-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://tensorflow.blog/llm-from-scratch"><img src="https://tensorflowkorea.wordpress.com/wp-content/uploads/2025/09/ebb091ebb094eb8ba5llm_ebb3b8ecb185_ec959eeba9b4.jpg" width="100px"></a>
</td>
</tr>
</table>

# 챕터 6: 텍스트 분류를 위한 미세 튜닝


In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",  # 그래프 라이브러리
        "numpy",       # 파이토치와 텐서플로의 종속성
        "tiktoken",    # 토크나이저
        "torch",       # 딥러닝 라이브러리
        # "tensorflow",  # OpenAI의 사전 훈련된 가중치를 위해서
        "pandas"       # 데이터셋 로딩을 위해
       ]
for p in pkgs:
    print(f"{p} 버전: {version(p)}")

<img src="images/llm_from_scratch/ch06_compressed/01.webp" width=800px>


## 6.1 여러 가지 미세 튜닝 방법


- 가장 일반적인 언어 모델 미세 튜닝 방법은 지시 미세 조정 및 분류 미세 튜닝입니다.
- 아래에 나타난 지시 미세 튜닝은 다음 장의 주제입니다.
- pretrained model에서 finetuning하는 것이 효과적


<img src="images/llm_from_scratch/ch06_compressed/02.webp" width=700px>


- 이 장의 주제인 분류 미세 튜닝(Classification finetuning)은 머신 러닝에 대한 배경 지식이 있다면 이미 익숙할 수 있는 절차입니다. 예를 들어, 손글씨 숫자를 분류하기 위해 합성곱 신경망을 훈련하는 것과 유사합니다.
- 분류 미세 튜닝에서는 모델이 출력할 수 있는 특정 개수의 클래스 레이블(예: "스팸" 및 "스팸아님")이 있습니다.
- 분류 미세 튜닝된 모델은 훈련 중에 본 클래스(예: "스팸" 또는 "스팸아님")만 예측할 수 있는 반면, 지시 미세 튜닝된 모델은 일반적으로 많은 작업을 수행할 수 있습니다.
- 분류 미세 튜닝된 모델을 매우 특수화된 모델로 생각할 수 있습니다. 실제로는 다양한 작업에서 잘 수행되는 일반 모델보다 특수 모델을 만드는 것이 훨씬 쉽습니다.


<img src="images/llm_from_scratch/ch06_compressed/03.webp" width=600px>


## 6.2 데이터셋 준비


<img src="images/llm_from_scratch/ch06_compressed/04.webp" width=700px>


- 이 섹션에서는 분류 미세 튜닝에 사용할 데이터셋을 준비합니다.
- 스팸 및 스팸아님 텍스트 메시지로 구성된 데이터셋을 분류하기 위해 LLM을 미세 튜닝합니다.
- 먼저 데이터셋을 다운로드하고 압축을 풉니다.


In [ ]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "datas/sms_spam_collection.zip"
extracted_path = "datas/sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path}가 이미 있어 다운로드 및 압축 해제를 건너뜁니다.")
        return

    # 파일을 다운로드 합니다.
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    # 파일 압축을 풉니다.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # .tsv 파일 확장자를 추가합니다.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}에 저장되었습니다.")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"기본 URL 실패: {e}. 백업 URL을 시도합니다...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


- 데이터셋은 탭으로 구분된 텍스트 파일로 저장되며, Pandas DataFrame으로 로드할 수 있습니다.


In [ ]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

- 클래스 분포를 확인해 보면, 데이터에 "스팸"보다 "햄"(즉, "스팸아님")이 훨씬 더 많이 포함되어 있음을 알 수 있습니다.


In [ ]:
print(df["Label"].value_counts())

- 간단하게 하기 위해, 그리고 교육 목적으로 작은 데이터셋을 선호하기 때문에 (LLM을 더 빠르게 미세 튜닝할 수 있음), 각 클래스마다 747개의 샘플이 포함되도록 데이터 세트를 서브샘플링(언더샘플링)합니다.
- (언더샘플링 외에도 클래스 불균형을 처리하는 여러 가지 다른 방법이 있지만 LLM에 관한 책의 범위를 벗어납니다. [`imbalanced-learn` 사용자 가이드](https://imbalanced-learn.org/stable/user_guide.html)에서 예제와 자세한 정보를 찾을 수 있습니다.)
- 데이터가 편향되어 있으면
  - 다수 클래스로 편향이 됨
  - 정확도가 높지만 스팸을탐ㅌ지하지 못할 가능성이 있음
  - 스팸을 노이지로 취급해 무시해 버릴 가능성이 있음

In [ ]:
def create_balanced_dataset(df):

    # "스팸" 샘플 개수 세기
    num_spam = df[df["Label"] == "spam"].shape[0]

    # "스팸" 샘플 개수와 일치하도록 "햄" 샘플을 무작위로 샘플링
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    # "햄"과 "스팸"을 합침
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

- 다음으로, 문자열 클래스 레이블 "ham"과 "spam"을 정수 클래스 레이블 0과 1로 변경합니다.


In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
balanced_df

- 데이터셋을 훈련 세트, 검증 세트, 테스트 세트로 무작위 분할하는 함수를 정의해 보겠습니다.


In [ ]:
import pandas as pd

def random_split(df, train_frac, validation_frac):
    """
    데이터프레임을 학습, 검증, 테스트 세트로 무작위 분할하는 함수
    
    Args:
        df: 분할할 전체 데이터프레임
        train_frac: 학습 데이터 비율 (예: 0.7)
        validation_frac: 검증 데이터 비율 (예: 0.1)
        # 테스트 데이터 비율은 남은 비율로 자동 결정됩니다.
    """
    
    # 1. 데이터프레임 전체 섞기 (Shuffle)
    # - frac=1: 전체 데이터의 100%를 샘플링 (모든 데이터를 가져오되 순서만 섞음)
    # - random_state=123: 매번 실행할 때마다 똑같은 순서로 섞이도록 고정 (재현성 확보)
    # - reset_index(drop=True): 섞인 후 뒤죽박죽된 인덱스를 버리고, 0부터 다시 차례대로 부여
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # 2. 분할할 지점(인덱스 위치) 계산
    # 전체 개수(len)에 비율을 곱해 어디서 자를지 정수형(int)으로 계산
    train_end = int(len(df) * train_frac)                          # 학습 데이터가 끝나는 지점
    validation_end = train_end + int(len(df) * validation_frac)    # 검증 데이터가 끝나는 지점

    # 3. 데이터프레임 슬라이싱 (Slicing)으로 분할
    train_df = df[:train_end]                     # 처음부터 ~ 학습 끝 지점까지
    validation_df = df[train_end:validation_end]  # 학습 끝 지점부터 ~ 검증 끝 지점까지
    test_df = df[validation_end:]                 # 검증 끝 지점부터 ~ 마지막 끝까지 (나머지 전부)

    return train_df, validation_df, test_df

# 함수 실행 예시
# balanced_df라는 데이터를 7:1:2 비율로 나눕니다.
# (Train: 70%, Validation: 10%, Test: 나머지 20%)
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

# 4. 결과 저장 (CSV 파일 생성)
# index=None: 저장할 때 불필요한 인덱스 번호(0, 1, 2...)는 파일에 포함하지 않음
train_df.to_csv("datas/train.csv", index=None)
validation_df.to_csv("datas/validation.csv", index=None)
test_df.to_csv("datas/test.csv", index=None)

## 6.3 데이터 로더 만들기


- 텍스트 메시지의 길이가 다르다는 점에 유의하세요. 배치에 여러 훈련 샘플을 결합하려면 다음 중 하나를 수행해야 합니다.
  1. 데이터셋 또는 배치에서 가장 짧은 메시지 길이로 모든 메시지를 자릅니다.
  2. 데이터셋 또는 배치에서 가장 긴 메시지 길이로 모든 메시지에 패딩을 추가합니다.

- 옵션 2를 선택하고 데이터셋에서 가장 긴 메시지에 맞춰 모든 메시지에 패딩을 추가합니다.
- 이를 위해 2장에서 설명한 대로 `<|endoftext|>`를 패딩 토큰으로 사용합니다.


<img src="images/llm_from_scratch/ch06_compressed/06.webp" width=800px>


In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

- `SpamDataset` 클래스는 학습 데이터셋에서 가장 긴 시퀀스를 식별하고 다른 시퀀스에 패딩 토큰을 추가하여 해당 길이에 맞춥니다.


In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd  # 데이터프레임 사용을 위해 pandas 임포트 필요

class SpamDataset(Dataset):
    """
    CSV 파일의 텍스트 데이터를 읽어 토큰화, 자르기, 패딩을 수행하여
    PyTorch 모델에 입력 가능한 형태로 변환하는 데이터셋 클래스
    """
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        # 1. 데이터 로드: pandas를 사용하여 CSV 파일을 읽어옵니다.
        self.data = pd.read_csv(csv_file)

        # 2. 토큰화 (Tokenization):
        # 데이터프레임의 "Text" 컬럼에 있는 모든 문장을 하나씩 꺼내서
        # tokenizer.encode()를 통해 정수 리스트(숫자)로 변환합니다.
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        # 3. 최대 길이(max_length) 결정 로직:
        if max_length is None:
            # 사용자가 길이를 지정하지 않았다면, 데이터 중 가장 긴 문장의 길이를 사용
            self.max_length = self._longest_encoded_length()
        else:
            # 사용자가 길이를 지정했다면 그 값을 사용하고,
            self.max_length = max_length
            # 지정된 길이보다 긴 문장은 뒷부분을 잘라냅니다 (Truncation).
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 4. 패딩 (Padding):
        # 모든 문장의 길이를 self.max_length로 통일합니다.
        # 문장 길이가 max_length보다 짧다면, 부족한 만큼 pad_token_id로 채웁니다.
        # 예: 문장 [10, 20], max=4, pad=0 -> [10, 20, 0, 0]
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        """
        데이터셋에서 특정 인덱스(index)의 샘플을 가져오는 함수
        데이터 로더(DataLoader)가 배치를 만들 때 이 함수를 반복 호출합니다.
        """
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]

        # 파이토치 모델은 '텐서(Tensor)' 형태만 입력받을 수 있으므로 변환합니다.
        # dtype=torch.long은 정수형(인덱스, 라벨 등)을 의미합니다.
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        # 데이터셋의 전체 샘플 개수를 반환 (학습 시 에포크 계산 등에 사용됨)
        return len(self.data)

    def _longest_encoded_length(self):
        # 데이터셋 내의 모든 문장 중 가장 긴 문장의 길이를 찾는 헬퍼 함수
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        
        # (참고) 파이썬의 max() 함수를 쓰면 위 6줄을 한 줄로 줄일 수 있습니다:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [ ]:
train_dataset = SpamDataset(
    csv_file="datas/train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

- 또한 검증 세트와 테스트 세트를 가장 긴 훈련 시퀀스에 맞춰 패딩합니다.
- 검증 세트와 테스트 세트 샘플 중 가장 긴 훈련 세트 샘플보다 긴 샘플은 `SpamDataset` 코드에서 `encoded_text[:self.max_length]`를 통해 잘린다는 점에 유의하세요.
- 이 동작은 완전히 선택 사항이며, 검증 세트와 테스트 세트의 경우 모두 `max_length=None`으로 설정해도 잘 작동합니다.


In [ ]:
val_dataset = SpamDataset(
    csv_file="datas/validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="datas/test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

- 다음으로, 데이터셋을 사용하여 데이터 로더를 만듭니다. 이는 이전 장에서 데이터 로더를 생성하는 것과 유사합니다.


<img src="images/llm_from_scratch/ch06_compressed/07.webp" width=700px>


In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True, #학습할 때는 shuffle가 중요
    num_workers=num_workers,
    drop_last=True, #미지막을 버릴지 말지
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- 검증 단계로서, 데이터 로더를 반복하고 각 배치에 120개의 토큰으로 구성된 8개의 훈련 샘플이 포함되어 있는지 확인합니다.


In [ ]:
print("훈련 세트 로더:")
for input_batch, target_batch in train_loader:
    pass

print("입력 배치 차원:", input_batch.shape)
print("레이블 배치 차원", target_batch.shape)

- 마지막으로, 각 데이터셋의 총 배치 수를 출력해 보겠습니다.


In [ ]:
print(f"{len(train_loader)}개 훈련 배치")
print(f"{len(val_loader)}개 검증 배치")
print(f"{len(test_loader)}개 테스트 배치")

## 6.4 사전 훈련된 가중치로 모델 초기화하기


- 이 섹션에서는 이전 챕터에서 작업했던 사전 훈련된 모델을 초기화합니다.

<img src="images/llm_from_scratch/ch06_compressed/08.webp" width=700px>


In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘사전 크기
    "context_length": 1024,  # 문맥 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12}, #트랜스포머 블록(Transformer Block)을 몇 층으로 쌓아 올렸는지
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"데이터셋 길이 {train_dataset.max_length}가 모델의 문맥 "
    f"길이 {BASE_CONFIG['context_length']}를 초과합니다. `max_length={BASE_CONFIG['context_length']}`로 "
    f"데이터 셋을 다시 초기화하십시오."
)

In [ ]:
from previous_chapters import GPTModel, load_gpt2_model

model_name = "gpt2-small-124M.pth"
model = load_gpt2_model(model_name, BASE_CONFIG)
model.eval()

- 모델이 제대로 로드되었는지 확인하기 위해 일관된 텍스트를 생성하는지 다시 한번 확인해 보겠습니다.


In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 모델을 분류기로 미세 튜닝하기 전에 프롬프트를 통해 스팸 메시지를 분류할 수 있는지 확인해 보겠습니다.


In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 보시다시피, 이 모델은 지시 사항을 잘 따르지 못합니다.
- 이는 모델이 사전 학습만 되었고 지시 미세 튜닝되지 않았기 때문에 예상되는 결과입니다 (지시 미세 튜닝은 다음 장에서 다룹니다).


## 6.5 분류 헤드 추가하기


<img src="images/llm_from_scratch/ch06_compressed/09.webp" width=700px>


- 이 절에서는 사전 훈련된 LLM을 분류 미세 튜닝에 사용할 수 있도록 수정합니다.
- 먼저 모델 구조를 살펴보겠습니다.(위의 그림과 연관지어서 구조 파악)


- 위에서 4장에서 구현한 구조가 깔끔하게 정리되어 있는 것을 볼 수 있습니다.
- 목표는 출력 층 교체하고 미세 튜닝하는 것입니다.
- 이를 위해 먼저 모델을 동결합니다. 즉, 모든 층을 훈련 불가능하게 만듭니다.


In [ ]:
for param in model.parameters():
    param.requires_grad = False

- 그런 다음 출력 층(`model.out_head`)을 교체합니다. 이 층은 입력을 50,257차원(어휘사전 크기)으로 매핑합니다.
- 이진 분류("스팸" 및 "스팸아님"의 두 클래스 예측)를 위해 모델을 미세 튜닝하기 때문에 아래와 같이 출력 층를 교체할 수 있으며 기본적으로 학습 가능합니다.
- 아래 코드를 더 일반적으로 유지하기 위해 `BASE_CONFIG["emb_dim"]`("gpt2-small (124M)" 모델에서 768과 같음)을 사용하는 것에 유의하세요.


In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

- 기술적으로 출력 층만 훈련하는 것으로 충분합니다.
- 하지만, [대규모 언어 모델 미세 튜닝](https://magazine.sebastianraschka.com/p/finetuning-large-language-models)에서 확인했듯이, 실험 결과 추가 층을 미세 튜닝하면 성능이 눈에 띄게 향상되는 것으로 나타났습니다.
- 따라서 마지막 트랜스포머 블록과 마지막 트랜스포머 블록을 출력 층에 연결하는 최종 `LayerNorm` 모듈도 훈련 가능하도록 설정합니다.


- 데이터가 적으면 layer를 늘린다고 정확도가 증가하지 않음

<img src="images/relation_with_layers_and_finetuninig.webp" width=500px>

<img src="images/llm_from_scratch/ch06_compressed/10.webp" width=700px>


- 최종 layer을 학습가능하도록 설정

In [ ]:
# 1. 마지막 트랜스포머 블록의 파라미터 잠금 해제
# model.trf_blocks[-1]: 트랜스포머 블록 리스트 중 '가장 마지막(-1)' 블록을 선택
for param in model.trf_blocks[-1].parameters():
    # requires_grad = True:
    # 이 파라미터(가중치)에 대해 기울기(Gradient)를 계산하도록 설정합니다.
    # 즉, 역전파(Backpropagation) 때 이 가중치를 '업데이트(학습)' 하겠다는 뜻입니다.
    param.requires_grad = True

# 2. 마지막 정규화(Normalization) 층의 파라미터 잠금 해제
# 모델의 최종 출력 직전에 있는 LayerNorm 층도 학습 대상에 포함시킵니다.
for param in model.final_norm.parameters():
    # 마찬가지로 이 층의 가중치도 학습되도록 설정합니다.
    param.requires_grad = True

- 이전 장에서처럼 이 모델을 비슷하게 사용할 수 있습니다.
- 예를 들어, 텍스트 입력을 제공해 보겠습니다.


In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("입력:", inputs)
print("입력 차원:", inputs.shape) # shape: (배치 크기, 토큰 수)

- 이전 장과 다른 점은 출력 차원이 50,257개가 아니라 2개라는 것입니다.


In [ ]:
with torch.no_grad():
    outputs = model(inputs)

print("출력:\n", outputs)
print("출력 차원:", outputs.shape) # shape: (배치 크기, 토큰 수, 클래스 수)

- 이전 장에서 논의했듯이 각 입력 토큰에 대해 하나의 출력 벡터가 있습니다.
- 모델에 4개의 입력 토큰이 있는 텍스트 샘플을 제공했으므로 출력은 2차원 출력 벡터 4개로 구성됩니다.


<img src="images/llm_from_scratch/ch06_compressed/11.webp" width=800px>


- 3장에서는 각 입력 토큰을 다른 모든 입력 토큰에 연결하는 어텐션 메커니즘에 대해 논의했습니다.
- 3장에서는 GPT와 같은 모델에 사용되는 코잘 어텐션 마스크도 소개했습니다. 코잘 마스크는 현재 토큰이 현재 및 이전 토큰 위치만 참조하도록 만듭니다.
- 코잘 어텐션 메커니즘을 기반으로, 4번째 (마지막) 토큰이 다른 모든 토큰에 대한 정보를 포함하는 유일한 토큰이기 때문에 모든 토큰 중에서 가장 많은 정보를 담고 있습니다.
- 따라서 우리는 이 마지막 토큰에 특히 관심이 있으며, 스팸 분류 작업을 위해 미세 튜닝할 것입니다.


In [ ]:
print("마지막 출력 토큰:", outputs[:, -1, :])

<img src="images/llm_from_scratch/ch06_compressed/12.webp" width=400px>


## 6.6 분류 손실과 정확도 계산하기


<img src="images/llm_from_scratch/ch06_compressed/13.webp" width=700px>


- 손실 계산을 설명하기 전에 모델 출력이 어떻게 클래스 레이블로 변환되는지 간략하게 살펴보겠습니다.


<img src="images/llm_from_scratch/ch06_compressed/14.webp" width=800px>


- 5장과 유사하게, `softmax` 함수를 통해 출력(로짓)을 확률 점수로 변환한 다음 `argmax` 함수를 통해 가장 큰 확률 값의 인덱스를 얻습니다.


In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("클래스 레이블:", label.item())

- 5장에서 설명했듯이 소프트맥스 함수는 여기서는 선택 사항입니다. 가장 큰 출력이 가장 큰 확률 점수에 해당하기 때문입니다.


In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("클래스 레이블:", label.item())

- 이 개념을 적용하여 소위 분류 정확도를 계산할 수 있습니다. 분류 정확도는 주어진 데이터셋에서 정확하게 예측한 백분율입니다.
- 분류 정확도를 계산하기 위해 앞서 설명한 `argmax` 기반 예측 코드를 데이터셋에 있는 모든 샘플에 적용하고 다음과 같이 정확한 예측의 비율을 계산할 수 있습니다.


In [ ]:
import torch

def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    """
    데이터 로더를 순회하며 모델의 정확도를 계산하는 함수
    """
    # 1. 평가 모드 전환 (매우 중요)
    # Dropout이나 Batch Normalization 같이 학습 때만 동작해야 하는 기능들을 비활성화합니다.
    model.eval()
    
    correct_predictions, num_examples = 0, 0

    # 2. 평가할 배치 개수 설정
    # num_batches가 없으면 전체 데이터를, 있으면 지정된 개수만큼만 확인합니다 (빠른 테스트용).
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
        
    # 3. 데이터 로더 순회 (enumerate로 인덱스 i와 데이터를 같이 받음)
    for i, (input_batch, target_batch) in enumerate(data_loader):
        # 지정된 배치 개수까지만 반복
        if i < num_batches:
            # 데이터를 GPU(또는 지정된 device)로 이동
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            # 4. 기울기(Gradient) 계산 비활성화 context
            # 평가만 할 것이므로 역전파를 위한 기록을 하지 않아 메모리를 아끼고 속도를 높입니다.
            with torch.no_grad():
                # 모델 예측 수행
                # 결과 shape 가정: (Batch_Size, Sequence_Length, Num_Classes)
                # [:, -1, :] -> 모든 배치의(:), 가장 마지막 토큰(-1)의, 모든 클래스 확률(:)을 가져옴
                logits = model(input_batch)[:, -1, :] 
            
            # 5. 예측값 결정
            # dim=-1: 가장 마지막 차원(클래스 확률)에서 가장 큰 값의 인덱스(클래스 번호)를 찾음
            predicted_labels = torch.argmax(logits, dim=-1)

            # 6. 정확도 누적 계산
            # 현재 배치의 데이터 개수 더하기
            num_examples += predicted_labels.shape[0]
            # 예측값과 정답이 같은지 비교(==)하여 True(1)의 개수(.sum)를 더함
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
            
    # 전체 맞은 개수 / 전체 데이터 개수 = 정확도
    return correct_predictions / num_examples

- 현재 모델 파라미터는 CPU에 로딩되어 있음

In [ ]:
devices = {param.device for param in model.parameters()}
print(devices)

- 여러 데이터셋에 대한 분류 정확도를 계산하기 위해 함수를 적용해 보겠습니다.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"실행 장치: {device}")

model.to(device) # nn.Module 클래스의 경우 model = model.to(device) 할당문이 필요하지 않습니다.

torch.manual_seed(123) # 데이터 로더에서 셔플링하기 때문에 재현성을 위해

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

- 예상대로 모델을 미세 튜닝하지 않았기 때문에 예측 정확도가 그리 좋지 않습니다.


- 미세 튜닝(훈련)을 시작하기 전에 먼저 훈련 중에 최적화하려는 손실 함수를 정의해야 합니다.
- 목표는 모델의 스팸 분류 정확도를 최대화하는 것이지만 분류 정확도는 미분 가능한 함수가 아닙니다.
- 따라서 분류 정확도를 최대화하는 대신, 크로스 엔트로피 손실을 최소화합니다 (무료 강의 [Introduction to Deep Learning](https://sebastianraschka.com/blog/2021/dl-course.html#l08-multinomial-logistic-regression--softmax-regression) 8강에서 이 주제에 대해 자세히 알아볼 수 있습니다)

- `calc_loss_batch` 함수는 5장과 동일하지만 모든 토큰 `model(input_batch)` 대신 마지막 토큰 `model(input_batch)[:, -1, :]`을 최적화하는 데에만 관심이 있습니다.


In [ ]:
import torch

def calc_loss_batch(input_batch, target_batch, model, device):
    """
    배치 단위로 모델의 손실(Loss)을 계산하는 함수
    """
    # 1. 데이터를 연산 장치(GPU 또는 CPU)로 이동
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)

    # 2. 모델 예측 (Forward Pass)
    # logits shape: [배치 크기, 문장 길이, 단어 집합 크기(Vocab Size)]
    logits = model(input_batch)

    # 3. 손실 계산 방식 분기 (중요!)
    # target_batch의 차원 수(dim)를 확인하여 작업 종류를 구분합니다.
    
    # CASE A: 단순 분류 (Sequence Classification)
    # 정답(target)이 [0, 1, 0] 처럼 1차원인 경우 (각 문장마다 정답이 하나)
    if target_batch.dim() == 1:
        # 문장의 '가장 마지막 토큰'의 예측값(logits[:, -1, :])만 사용하여
        # 정답과 비교해 손실을 계산합니다.
        loss = torch.nn.functional.cross_entropy(logits[:, -1, :], target_batch)
        
    # CASE B: 언어 모델링 (Language Modeling / Next Token Prediction)
    # 정답이 [[10, 20], [30, 40]] 처럼 2차원인 경우 (모든 단어 위치마다 정답이 존재)
    else:
        # 모델 출력과 정답을 일렬로 쭉 폅니다(Flatten).
        # logits: [배치*길이, 단어집합크기], target: [배치*길이]
        # 이렇게 모양을 맞춰줘야 CrossEntropy가 모든 단어에 대해 오차를 평균 낼 수 있습니다.
        loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

    return loss

`calc_loss_loader`는 5장과 완전히 동일합니다.


In [ ]:
# 5장과 동일
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # num_batches가 데이터 로더의 배치 수를 초과하는 경우
        # 데이터 로더의 총 배치 수와 일치하도록 배치 수를 줄입니다.
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- `calc_closs_loader`를 사용하여 훈련을 시작하기 전 초기 훈련, 검증 및 테스트 세트 손실을 계산합니다.


In [ ]:
with torch.no_grad(): # 효율성을 위해 그레이디언트 추적을 비활성화합니다. 아직 훈련하지 않기 때문입니다.
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"훈련 손실: {train_loss:.3f}")
print(f"검증 손실: {val_loss:.3f}")
print(f"테스트 손실: {test_loss:.3f}")

- 다음 섹션에서는 손실 값과 결과적으로 분류 정확도를 개선하기 위해 모델을 훈련합니다.


## 6.7 지도 학습 데이터로 모델 미세 튜닝하기


- 이 절에서는 모델의 분류 정확도를 향상시키기 위한 훈련 함수를 정의하고 사용합니다.
- 아래 `train_classifier_simple` 함수는 5장에서 모델 사전 훈련에 사용했던 `train_model_simple` 함수와 거의 동일합니다.
- 단 두 가지 차이점은 다음과 같습니다.
  1. 토큰 수 대신 학습된 샘플 수(`examples_seen`)를 추적합니다.
  2. 각 에포크 다음에 샘플 텍스트를 출력하는 대신 정확도를 계산합니다.


<img src="images/llm_from_scratch/ch06_compressed/15.webp" width=600px>


In [ ]:
import torch

# 5장의 `train_model_simple`과 구조가 거의 같지만, 분류(Classification)를 위해 일부 수정된 함수
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    """
    모델을 학습시키고, 주기적으로 검증 데이터로 성능을 평가하는 함수
    
    Args:
        model: 학습할 파이토치 모델
        train_loader: 학습 데이터 공급자 (DataLoader)
        val_loader: 검증 데이터 공급자 (DataLoader)
        optimizer: 가중치를 업데이트하는 최적화 도구 (예: AdamW)
        device: 학습을 진행할 장치 (CPU or GPU)
        num_epochs: 전체 데이터를 몇 번 반복해서 학습할지
        eval_freq: 몇 번의 스텝(배치)마다 손실을 출력할지 설정
        eval_iter: 평가 시 몇 개의 배치만 사용할지 (속도를 위해 일부만 사용 가능)
    """
    
    # 1. 기록용 리스트 초기화 (나중에 그래프를 그리기 위해 저장)
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # 2. 메인 학습 루프 (에포크 단위 반복)
    for epoch in range(num_epochs):
        model.train()  # 중요: 모델을 '학습 모드'로 전환 (Dropout 등이 켜짐)

        # 3. 배치 단위 반복 (데이터 로더에서 배치를 하나씩 꺼냄)
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # (a) 기울기 초기화: 이전 배치의 계산 찌꺼기 제거
            
            # (b) 손실(Loss) 계산: 모델이 예측하고, 정답과 비교하여 오차 계산
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            
            # (c) 역전파(Backpropagation): 오차를 줄이는 방향(기울기) 계산
            loss.backward() 
            
            # (d) 가중치 업데이트: 계산된 방향으로 모델 파라미터 수정
            optimizer.step() 

            # 진행 상황 추적
            examples_seen += input_batch.shape[0] # 몇 개의 데이터를 봤는지 누적
            global_step += 1

            # 4. 중간 평가 단계 (eval_freq 스텝마다 실행)
            # 학습 중간중간에 손실(Loss)이 잘 줄어들고 있는지 확인
            if global_step % eval_freq == 0:
                # evaluate_model 함수를 호출하여 현재 상태의 손실 계산
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                
                print(f"에포크 {epoch+1} (Step {global_step:06d}): "
                      f"훈련 손실 {train_loss:.3f}, 검증 손실 {val_loss:.3f}")

        # 5. 에포크 종료 후 정확도(Accuracy) 계산
        # 손실(Loss)만으로는 직관적이지 않으므로, '몇 퍼센트 맞췄는지' 확인
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        
        print(f"훈련 정확도: {train_accuracy*100:.2f}% | ", end="")
        print(f"검증 정확도: {val_accuracy*100:.2f}%")
        
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

- `evaluate_model` 함수는 `train_classifier_simple` 에서 사용되며 5장에서 사용한 것과 동일합니다.


In [ ]:
# 5장과 동일
# 모델의 손실을 평가하는 헬퍼 함수
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()  # 중요: 모델을 '평가 모드'로 전환 (Dropout 등이 꺼짐)
    
    # 평가 시에는 기울기 계산을 하지 않음 (메모리 절약, 속도 향상)
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    
    model.train() # 평가가 끝나면 다시 '학습 모드'로 복구
    
    return train_loss, val_loss

- M3 맥북 에어 노트북 컴퓨터에서는 훈련 시간이 약 5분 정도 소요되며 V100 또는 A100 GPU에서는 30초 미만이 소요됩니다.


In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분")

- 5장과 유사하게, 맷플롯립을 사용하여 훈련 및 검증 세트에 대한 손실 함수를 그립니다.


In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 훈련 및 검증 손실을 에포크에 따라 그립니다.
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # 처리한 샘플 수를 위해 두 번째 x축을 만듭니다.
    ax2 = ax1.twiny()  # 동일한 y축을 공유하는 두 번째 x축을 만듭니다.
    ax2.plot(examples_seen, train_values, alpha=0)  # 눈금 정렬을 위한 보이지 않는 그래프
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # 공간을 확보하기 위해 레이아웃을 조정합니다.
    plt.savefig(f"outputs/{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

- 기울기가 아래쪽으로 향하고 있으므로 모델이 잘 학습하고 있음을 알 수 있습니다.
- 또한, 훈련 손실과 검증 손실이 매우 가깝다는 사실은 모델이 훈련 데이터에 과대적합되는 경향이 없음을 나타냅니다.
- 마찬가지로, 아래에서 정확도를 그래프로 표시할 수 있습니다.


In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

- 위의 정확도 그래프를 기반으로 에포크 4와 5 이후 모델이 비교적 높은 훈련 및 검증 정확도를 달성했음을 알 수 있습니다.
- 하지만 이전에 훈련 함수에서 `eval_iter=5`를 지정했던 것을 기억해야 합니다. 이는 훈련 및 검증 세트의 일부만 사용해 성능을 추정했음을 의미합니다.
- 아래와 같이 전체 데이터셋에 대한 훈련, 검증 및 테스트 세트 성능을 계산할 수 있습니다.


In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

- 훈련 세트와 검증 세트의 성능이 거의 동일한 것을 확인할 수 있습니다.
- 그러나 테스트 세트 성능이 약간 낮은 것을 보면 모델이 훈련 데이터에 대해 아주 작은 정도로 과적합되었음을 알 수 있으며, 학습률과 같은 일부 하이퍼파라미터를 조정하는 데 사용된 검증 데이터에도 과적합되었음을 알 수 있습니다.
- 하지만 이는 정상적인 현상이며, 모델의 드롭아웃 비율(`drop_rate`)이나 옵티마이저 설정의 `weight_decay`를 높여 이러한 차이를 줄일 수 있습니다.


## 6.8 LLM을 스팸 분류기로 사용하기


<img src="images/llm_from_scratch/ch06_compressed/18.webp" width=700px>


- 마지막으로, 미세 튜닝된 GPT 모델을 실제로 사용해 보겠습니다.
- 아래 `classify_review` 함수는 이전에 구현한 `SpamDataset`과 유사한 데이터 전처리 단계를 구현합니다.
- 그런 다음, 이 함수는 모델에서 예측된 정수 클래스 레이블을 반환하고 해당 클래스 이름을 반환합니다.


In [ ]:
import torch

def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    """
    단일 텍스트(리뷰)를 입력받아 스팸 여부를 분류하는 함수
    """
    # 1. 평가 모드 설정
    # 학습이 아니므로 Dropout 등을 비활성화하여 일관된 결과를 얻습니다.
    model.eval()

    # 2. 텍스트 토큰화 (Encoding)
    # 문자열(text)을 모델이 이해하는 정수 리스트(input_ids)로 변환합니다.
    input_ids = tokenizer.encode(text)
    
    # 모델이 물리적으로 처리할 수 있는 최대 길이 확인 (예: GPT-2는 보통 1024)
    supported_context_length = model.pos_emb.weight.shape[0]

    # 3. 시퀀스 자르기 (Truncation)
    # 입력이 너무 길면 모델이 처리할 수 없으므로 잘라냅니다.
    # 사용자가 설정한 max_length와 모델 한계치 중 더 작은 값으로 길이를 제한합니다.
    # 주의: 아래 코드는 max_length가 None일 경우 에러가 날 수 있으므로, assert문으로 방어합니다.
    input_ids = input_ids[:min(max_length, supported_context_length)]
    
    # (안전장치) max_length가 올바르게 설정되었는지 확인
    assert max_length is not None, (
        "max_length가 지정되지 않았습니다. 모델의 최대 문맥 길이를 사용하려면 "
        "max_length=model.pos_emb.weight.shape[0]로 지정하세요."
    )
    assert max_length <= supported_context_length, (
        f"max_length({max_length})가 모델이 지원하는 문맥 길이({supported_context_length})를 초과했습니다."
    )
    
    # 4. 패딩 (Padding)
    # 설정한 max_length보다 문장이 짧을 경우, 남은 뒷부분을 pad_token_id로 채웁니다.
    # 모델은 고정된 길이의 입력을 선호하기 때문입니다.
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    
    # 5. 텐서 변환 및 배치 차원 추가 (매우 중요!)
    # 현재 input_ids는 [길이] 형태의 1차원 리스트입니다.
    # 하지만 모델은 [배치 크기, 길이] 형태의 2차원 입력을 기대합니다.
    # unsqueeze(0)을 통해 [1, 길이] 형태로 차원을 하나 늘려줍니다. (가짜 배치 생성)
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) 

    # 6. 모델 추론 (Inference)
    with torch.no_grad(): # 기울기 계산 끄기 (메모리 절약)
        # 모델에 입력을 넣어 출력(Logits)을 얻습니다.
        # [:, -1, :] : (모든 배치, 마지막 토큰, 모든 클래스) -> 분류를 위해 마지막 토큰만 사용
        logits = model(input_tensor)[:, -1, :]
    
    # 7. 결과 해석
    # argmax: 확률값 중 가장 큰 값의 인덱스(0 또는 1)를 뽑습니다.
    # item(): 텐서 값을 파이썬 정수(int)로 변환합니다.
    predicted_label = torch.argmax(logits, dim=-1).item()

    # 8. 사람이 읽을 수 있는 결과 반환
    # 1이면 "스팸", 0이면 "스팸아님" 리턴
    return "스팸" if predicted_label == 1 else "스팸아님"

- 몇 개의 샘플로 시험해 보겠습니다.


In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(classify_review(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

- 마지막으로, 나중에 다시 훈련하지 않고 모델을 재사용할 수 있도록 모델을 저장합니다.


In [ ]:
torch.save(model.state_dict(), "outputs/review_classifier.pth")

- 그런 다음 새 세션에서 다음과 같이 모델을 로드할 수 있습니다.


In [ ]:
model_state_dict = torch.load("outputs/review_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

## 요약


- 연습 문제 풀이는 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)에서 찾을 수 있습니다.
- 또한 관심 있는 독자는 [부록 E](../../appendix-E)에서 LoRA를 사용한 파라미터 효율적인 훈련에 대한 소개를 찾을 수 있습니다.
